# SIH 2026 - Problem Statement 26145: AI-Based Detection of Cyber Threats in Unidirectional IP Traffic
## Stage 1: Dataset Exploration (CIC-IDS2017)

This notebook performs a memory-safe exploratory analysis of the raw CIC-IDS2017 network traffic dataset.
It discovers all raw CSV partition files, inspects the schema and data structure, identifies target labels, and analyzes data quality (missing values and duplicate records) using a lightweight sample to preserve system memory (8 GB RAM target).

### A. Import Libraries
Import the necessary standard and data processing libraries (`pandas`, `pathlib`, `numpy`).

In [1]:
import os
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Filter warnings for cleaner presentation
warnings.filterwarnings('ignore')

# Display settings for clear, readable outputs
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Pandas version: 3.0.6
NumPy version: 2.5.3


### B. Locate the CIC-IDS2017 CSV Files
Search recursively inside `data/raw/MachineLearningCSV/` to identify all dataset partition files and display their sizes in MB without hardcoding filenames.

In [2]:
# Resolve dataset directory relative to project root or current working directory
possible_dirs = [
    Path('data/raw/MachineLearningCSV'),
    Path('../data/raw/MachineLearningCSV'),
    Path.cwd() / 'data' / 'raw' / 'MachineLearningCSV',
    Path.cwd().parent / 'data' / 'raw' / 'MachineLearningCSV'
]

dataset_dir = None
for p in possible_dirs:
    if p.exists() and p.is_dir():
        dataset_dir = p.resolve()
        break

if dataset_dir is None:
    print(f"❌ ERROR: Dataset directory not found in candidate paths: {[str(p) for p in possible_dirs]}")
    csv_files = []
    total_size_mb = 0.0
else:
    # Recursively find all CSV files
    csv_files = sorted(list(dataset_dir.rglob('*.csv')))

print(f"Dataset Directory: {dataset_dir}")
print(f"Number of CSV files: {len(csv_files)}\n")

if not csv_files:
    print("❌ ERROR: No CSV files found inside the dataset directory. Please verify that the dataset is extracted.")
    total_size_mb = 0.0
else:
    print(f"{'#':<3} | {'Filename':<55} | {'File Size (MB)':>15}")
    print("-" * 78)
    total_size_mb = 0.0
    for idx, f in enumerate(csv_files, 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        total_size_mb += size_mb
        print(f"{idx:<3} | {f.name:<55} | {size_mb:>12.2f} MB")
    print("-" * 78)
    print(f"Total Approximate Size: {total_size_mb:.2f} MB ({total_size_mb / 1024:.2f} GB)")

Dataset Directory: C:\Users\Anushka\.gemini\antigravity-ide\scratch\PS26145-CyberThreatDetector\data\raw\MachineLearningCSV
Number of CSV files: 8

#   | Filename                                                |  File Size (MB)
------------------------------------------------------------------------------
1   | Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        |        73.55 MB
2   | Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    |        73.34 MB
3   | Friday-WorkingHours-Morning.pcap_ISCX.csv               |        55.62 MB
4   | Monday-WorkingHours.pcap_ISCX.csv                       |       168.73 MB
5   | Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv |        79.25 MB
6   | Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  |        49.61 MB
7   | Tuesday-WorkingHours.pcap_ISCX.csv                      |       128.82 MB
8   | Wednesday-workingHours.pcap_ISCX.csv                    |       214.74 MB
-------------------------------------------------

### C. Inspect the Dataset Safely
Load a 1,000-row sample from the first discovered CSV file to inspect schema and structure safely without exhausting memory.
Display the filename, sample dimensions, column names, preview rows, and DataFrame info.

In [3]:
sample_df = None
sample_file = None

if not csv_files:
    print("⚠️ Skipping inspection because no CSV files were found.")
else:
    sample_file = csv_files[0]
    try:
        sample_df = pd.read_csv(sample_file, nrows=1000)
        print(f"Filename: {sample_file.name}")
        print(f"Number of rows in sample: {len(sample_df)}")
        print(f"Number of columns: {len(sample_df.columns)}")
        
        print("\n--- Column Names ---")
        for i, col in enumerate(sample_df.columns):
            print(f"{i+1:2d}. {repr(col)}")
        
        print("\n--- First 5 Rows Preview ---")
        display(sample_df.head())
        
        print("\n--- DataFrame Info ---")
        sample_df.info(verbose=True, show_counts=True)
    except Exception as e:
        print(f"❌ Error reading CSV file '{sample_file}': {e}")

Filename: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Number of rows in sample: 1000
Number of columns: 79

--- Column Names ---
 1. ' Destination Port'
 2. ' Flow Duration'
 3. ' Total Fwd Packets'
 4. ' Total Backward Packets'
 5. 'Total Length of Fwd Packets'
 6. ' Total Length of Bwd Packets'
 7. ' Fwd Packet Length Max'
 8. ' Fwd Packet Length Min'
 9. ' Fwd Packet Length Mean'
10. ' Fwd Packet Length Std'
11. 'Bwd Packet Length Max'
12. ' Bwd Packet Length Min'
13. ' Bwd Packet Length Mean'
14. ' Bwd Packet Length Std'
15. 'Flow Bytes/s'
16. ' Flow Packets/s'
17. ' Flow IAT Mean'
18. ' Flow IAT Std'
19. ' Flow IAT Max'
20. ' Flow IAT Min'
21. 'Fwd IAT Total'
22. ' Fwd IAT Mean'
23. ' Fwd IAT Std'
24. ' Fwd IAT Max'
25. ' Fwd IAT Min'
26. 'Bwd IAT Total'
27. ' Bwd IAT Mean'
28. ' Bwd IAT Std'
29. ' Bwd IAT Max'
30. ' Bwd IAT Min'
31. 'Fwd PSH Flags'
32. ' Bwd PSH Flags'
33. ' Fwd URG Flags'
34. ' Bwd URG Flags'
35. ' Fwd Header Length'
36. ' Bwd Header Length'
37. 'Fwd Packet

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,...,0,0,0.0000,0.0000,0,0,BENIGN
1,55054,109,1,1,6,6,6,...,0,0,0.0000,0.0000,0,0,BENIGN
2,55055,52,1,1,6,6,6,...,0,0,0.0000,0.0000,0,0,BENIGN
3,46236,34,1,1,6,6,6,...,0,0,0.0000,0.0000,0,0,BENIGN
4,54863,3,2,0,12,0,6,...,0,0,0.0000,0.0000,0,0,BENIGN



--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 79 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0    Destination Port             1000 non-null   int64  
 1    Flow Duration                1000 non-null   int64  
 2    Total Fwd Packets            1000 non-null   int64  
 3    Total Backward Packets       1000 non-null   int64  
 4   Total Length of Fwd Packets   1000 non-null   int64  
 5    Total Length of Bwd Packets  1000 non-null   int64  
 6    Fwd Packet Length Max        1000 non-null   int64  
 7    Fwd Packet Length Min        1000 non-null   int64  
 8    Fwd Packet Length Mean       1000 non-null   float64
 9    Fwd Packet Length Std        1000 non-null   float64
 10  Bwd Packet Length Max         1000 non-null   int64  
 11   Bwd Packet Length Min        1000 non-null   int64  
 12   Bwd Packet Length Mean       1000 non-null   floa

### D. Identify the Target Label Column
Identify whether the target column `Label` exists (accounting for potential whitespace padding) and inspect class distribution within the 1,000-row sample.

In [4]:
target_col = None

if sample_df is not None:
    # Check for 'Label' column (handling leading/trailing whitespace common in CIC-IDS2017)
    for col in sample_df.columns:
        if col.strip() == 'Label':
            target_col = col
            break

    if target_col is not None:
        print(f"✅ Found target label column: {repr(target_col)}")
        print(f"\nLabel Distribution (1,000-row sample):")
        label_dist = sample_df[target_col].value_counts(dropna=False)
        print(label_dist)
        print(f"\nNumber of unique labels in sample: {len(label_dist)}")
    else:
        print("⚠️ Warning: No column named 'Label' was found in the sample.")
        print(f"Available columns: {list(sample_df.columns)}")
else:
    print("⚠️ No sample data loaded.")

✅ Found target label column: ' Label'

Label Distribution (1,000-row sample):
 Label
BENIGN    1000
Name: count, dtype: int64

Number of unique labels in sample: 1


### E. Missing-Value Analysis
Calculate missing values per column, list columns with missing entries, and calculate total missing values across the sample.

In [5]:
if sample_df is not None:
    missing_per_col = sample_df.isnull().sum()
    cols_with_missing = missing_per_col[missing_per_col > 0]
    total_missing = int(missing_per_col.sum())

    print(f"Total missing values in sample: {total_missing}")
    print(f"Number of columns with missing values: {len(cols_with_missing)}")
    
    if len(cols_with_missing) > 0:
        print("\nColumns with Missing Values:")
        for col, count in cols_with_missing.items():
            pct = (count / len(sample_df)) * 100
            print(f" - {repr(col)}: {count} missing ({pct:.2f}%)")
    else:
        print("✅ No missing values detected in this 1,000-row sample.")
else:
    print("⚠️ No sample data loaded.")

Total missing values in sample: 0
Number of columns with missing values: 0
✅ No missing values detected in this 1,000-row sample.


### F. Duplicate Analysis
Calculate the number of duplicate rows present in the 1,000-row sample.

In [6]:
if sample_df is not None:
    duplicate_count = int(sample_df.duplicated().sum())
    print(f"Number of duplicate rows in sample: {duplicate_count}")
    if duplicate_count > 0:
        pct_dup = (duplicate_count / len(sample_df)) * 100
        print(f"Duplicate percentage in sample: {pct_dup:.2f}%")
    else:
        print("✅ No duplicate rows found in this 1,000-row sample.")
else:
    print("⚠️ No sample data loaded.")

Number of duplicate rows in sample: 0
✅ No duplicate rows found in this 1,000-row sample.


### G. Basic Numerical Inspection
Compute and display a compact descriptive-statistics table for numeric columns using `describe()`. Transposed summary view is used to maintain readability.

In [7]:
if sample_df is not None:
    numeric_df = sample_df.select_dtypes(include=[np.number])
    print(f"Number of numeric columns: {len(numeric_df.columns)} / {len(sample_df.columns)}")
    
    # Generate descriptive statistics
    desc = numeric_df.describe().T
    
    # Display compact summary: count, mean, std, min, median (50%), max
    compact_desc = desc[['count', 'mean', 'std', 'min', '50%', 'max']].rename(columns={'50%': 'median'})
    
    print("\n--- Compact Numerical Feature Statistics (First 15 Features) ---")
    display(compact_desc.head(15))
    
    print(f"\nDescriptive statistics computed for all {len(numeric_df.columns)} numeric columns.")
else:
    print("⚠️ No sample data loaded.")

Number of numeric columns: 78 / 79

--- Compact Numerical Feature Statistics (First 15 Features) ---


,count,mean,std,min,median,max
Destination Port,1000.0000,3423.3670,12493.3931,21.0000,80.0000,60954.0000
Flow Duration,1000.0000,8171073.0280,26664035.3249,0.0000,77439.0000,119952278.0000
Total Fwd Packets,1000.0000,16.1520,53.9793,1.0000,2.0000,1611.0000
Total Backward Packets,1000.0000,19.0120,65.4256,0.0000,2.0000,1935.0000
Total Length of Fwd Packets,1000.0000,641.6920,1600.6684,0.0000,94.0000,29739.0000
Total Length of Bwd Packets,1000.0000,30924.6170,117083.5099,0.0000,215.5000,3433707.0000
Fwd Packet Length Max,1000.0000,245.5350,391.5960,0.0000,50.0000,7148.0000
Fwd Packet Length Min,1000.0000,21.4980,23.1913,0.0000,6.0000,94.0000
Fwd Packet Length Mean,1000.0000,44.2766,60.3201,0.0000,37.0000,1326.4667
Fwd Packet Length Std,1000.0000,56.4318,111.4801,0.0000,0.0000,1830.9061



Descriptive statistics computed for all 78 numeric columns.


### H. Dataset Summary
Create a comprehensive summary table aggregating key findings from the exploratory inspection.

In [8]:
summary_data = {
    "Metric": [
        "Number of CSV files",
        "Total approximate size of CSV files",
        "Sample rows inspected",
        "Number of features/columns",
        "Target column",
        "Number of unique labels in the sample",
        "Duplicate count",
        "Total missing values"
    ],
    "Value": [
        len(csv_files) if 'csv_files' in locals() else 0,
        f"{total_size_mb:.2f} MB ({total_size_mb / 1024:.2f} GB)" if 'total_size_mb' in locals() else "N/A",
        len(sample_df) if sample_df is not None else 0,
        len(sample_df.columns) if sample_df is not None else 0,
        repr(target_col) if target_col is not None else "Not Found",
        len(sample_df[target_col].value_counts()) if (sample_df is not None and target_col is not None) else 0,
        duplicate_count if 'duplicate_count' in locals() else 0,
        total_missing if 'total_missing' in locals() else 0
    ]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)

,Metric,Value
0,Number of CSV files,8
1,Total approximate size of CSV files,843.66 MB (0.82 GB)
2,Sample rows inspected,1000
3,Number of features/columns,79
4,Target column,' Label'
5,Number of unique labels in the sample,1
6,Duplicate count,0
7,Total missing values,0


## Next Steps
- Clean and standardize the dataset
- Analyze attack classes
- Prepare datasets for threat-specific detection engines
- Split data for model training and evaluation